In [2]:
!pip install fastapi uvicorn pydantic pytorch-lightning -q

In [3]:
import os
import gc
import time
import json
import pickle
from pathlib import Path
import polars as pl
import lightgbm as lgb

import numpy as np
import torch
from torch import concat, diag, logical_and, logical_or, nn, tensor, tile
from torch.nn import Dropout
import pytorch_lightning as py_light
from functools import partial
import warnings
warnings.filterwarnings("ignore")

import threading
import time
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

In [4]:
TYPE_MAP = {
    "clicks": 0,
    "carts":  1,
    "orders": 2,
}

In [5]:
def map_type_sequence(type_sequence: Optional[List[str]]) -> Optional[List[int]]:
    # ánh xạ từ chuỗi hành động ký tự sang chuỗi số
    if type_sequence is None:
        return None
    invalid = [t for t in type_sequence if t not in TYPE_MAP]
    if invalid:
        raise ValueError(f"type_sequence chứa giá trị không hợp lệ: {invalid}. Chỉ chấp nhận: {list(TYPE_MAP.keys())}")
    return [TYPE_MAP[t] for t in type_sequence]

# LBMReRank


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. Các hàm Load Artifacts
# ══════════════════════════════════════════════════════════════════════════════

def load_pipeline_artifacts(artifacts_dir: str) -> tuple:
    """
    Tải lại toàn bộ các tài nguyên (ma trận, đặc trưng toàn cục, mô hình) từ đĩa.
    
    Args:
        artifacts_dir: Đường dẫn đến thư mục chứa các file lưu trữ.
        
    Returns:
        tuple: (covisit_clicks, covisit_cart_order, covisit_buy2buy, popular_items, 
                global_item_feats_all, global_item_feats_7d, models, feature_names)
    """
    dir_path = Path(artifacts_dir)
    if not dir_path.exists():
        raise FileNotFoundError(f"Không tìm thấy thư mục artifacts tại: {dir_path}")
        
    print(f"\n[Inference] Đang tải các tài nguyên từ {dir_path}...")
    t0 = time.time()
    
    covisit_clicks = pl.read_parquet(dir_path / "covisit_clicks.parquet")
    covisit_cart_order = pl.read_parquet(dir_path / "covisit_cart_order.parquet")
    covisit_buy2buy = pl.read_parquet(dir_path / "covisit_buy2buy.parquet")
    popular_items = pl.read_parquet(dir_path / "popular_items.parquet")
    global_item_feats_all = pl.read_parquet(dir_path / "global_item_feats_all.parquet")
    global_item_feats_7d = pl.read_parquet(dir_path / "global_item_feats_7d.parquet")
    
    with open(dir_path / "models.pkl", "rb") as f:
        models = pickle.load(f)
        
    with open(dir_path / "feature_names.json", "r") as f:
        feature_names = json.load(f)
        
    print(f"[Inference] Tải tài nguyên thành công trong {time.time() - t0:.1f}s!")
    return (
        covisit_clicks,
        covisit_cart_order,
        covisit_buy2buy,
        popular_items,
        global_item_feats_all,
        global_item_feats_7d,
        models,
        feature_names
    )

# ══════════════════════════════════════════════════════════════════════════════
# 2. Pipeline Core (Candidates & Feature Engineering)
# ══════════════════════════════════════════════════════════════════════════════

def generate_candidates(
    history_df: pl.DataFrame,
    df_clicks: pl.DataFrame,
    df_buys: pl.DataFrame,
    df_buy2buy: pl.DataFrame,
    popular_df: pl.DataFrame,
    chunk_size: int = 50_000,
) -> pl.DataFrame:
    """
    Tạo danh sách ứng viên (candidate retrieval) từ 5 nguồn:
    History, Popular items, Co-visitation clicks, Co-visitation cart-order, Co-visitation buy2buy.
    """
    all_sessions = history_df["session"].unique().sort().to_list()
    n_sessions = len(all_sessions)
    n_chunks = (n_sessions + chunk_size - 1) // chunk_size
    
    all_candidates = []
    
    for chunk_idx in range(n_chunks):
        start = chunk_idx * chunk_size
        end = min(start + chunk_size, n_sessions)
        chunk_sessions = all_sessions[start:end]
        
        # History cho chunk này
        history_chunk = history_df.filter(pl.col("session").is_in(chunk_sessions))
        
        # --- Nguồn 1: Lịch sử tương tác của user ---
        cand_history = (
            history_chunk
            .rename({"aid": "candidate_aid"})
            .with_columns(pl.lit(1).cast(pl.UInt8).alias("source_history"))
        )
        
        # --- Nguồn 2: Sản phẩm phổ biến ---
        sessions_in_chunk = history_chunk.select("session").unique()
        cand_popular = sessions_in_chunk.join(popular_df, how="cross")
        
        # --- Nguồn 3-5: Co-visitation matrices ---
        cand_clicks_raw = history_chunk.join(df_clicks, on="aid", how="inner")
        cand_buys_raw   = history_chunk.join(df_buys, on="aid", how="inner")
        cand_b2b_raw    = history_chunk.join(df_buy2buy, on="aid", how="inner")
        
        # --- Gộp tất cả ứng viên và lọc trùng trùng ---
        candidates_df = pl.concat([
            cand_history.select(["session", "candidate_aid"]),
            cand_popular.select(["session", "candidate_aid"]),
            cand_clicks_raw.select(["session", "candidate_aid"]),
            cand_buys_raw.select(["session", "candidate_aid"]),
            cand_b2b_raw.select(["session", "candidate_aid"]),
        ]).unique(subset=["session", "candidate_aid"])
        
        # --- Ghép lại các đặc trưng nguồn (source features) ---
        candidates_df = candidates_df.join(
            cand_history.select(["session", "candidate_aid", "source_history"]),
            on=["session", "candidate_aid"],
            how="left",
        )
        
        for cand_raw, name in [
            (cand_clicks_raw, "clicks"),
            (cand_buys_raw, "cart_order"),
            (cand_b2b_raw, "buy2buy"),
        ]:
            rank_col = f"rank_{name}"
            wgt_col = f"wgt_{name}"
            
            if rank_col in cand_raw.columns:
                agg = (
                    cand_raw
                    .group_by(["session", "candidate_aid"])
                    .agg([
                        pl.col(rank_col).min(),
                        pl.col(wgt_col).max(),
                    ])
                )
                candidates_df = candidates_df.join(
                    agg, on=["session", "candidate_aid"], how="left"
                )
        
        all_candidates.append(candidates_df)
        
    if not all_candidates:
        return pl.DataFrame(schema={
            "session": pl.Int64, "candidate_aid": pl.Int64, "source_history": pl.UInt8,
            "rank_clicks": pl.UInt16, "wgt_clicks": pl.Float32,
            "rank_cart_order": pl.UInt16, "wgt_cart_order": pl.Float32,
            "rank_buy2buy": pl.UInt16, "wgt_buy2buy": pl.Float32
        })
        
    result = pl.concat(all_candidates)
    return result


def create_session_features(df: pl.DataFrame) -> tuple:
    """Tạo các đặc trưng thống kê ở cấp độ session và sự tương tác."""
    # Session level
    session_feats = (
        df
        .group_by("session")
        .agg([
            pl.count().alias("session_length"),
            pl.col("aid").n_unique().alias("session_unique_aids"),
            pl.col("ts").max().alias("session_end_ts"),
            (pl.col("ts").max() - pl.col("ts").min()).alias("session_duration"),
            pl.col("type").filter(pl.col("type") == 0).count().alias("session_click_cnt"),
            pl.col("type").filter(pl.col("type") == 1).count().alias("session_cart_cnt"),
            pl.col("type").filter(pl.col("type") == 2).count().alias("session_order_cnt"),
        ])
    )
    
    session_feats = session_feats.with_columns([
        (pl.col("session_click_cnt") / (pl.col("session_length") + 1)).alias("session_click_ratio"),
        (pl.col("session_cart_cnt") / (pl.col("session_length") + 1)).alias("session_cart_ratio"),
    ])
    
    # Interaction level (per session-aid pair)
    interaction_feats = (
        df
        .group_by(["session", "aid"])
        .agg([
            pl.count().alias("num_repetitions"),
            pl.col("ts").max().alias("last_item_ts"),
            pl.col("ts").min().alias("first_item_ts"),
        ])
        .rename({"aid": "candidate_aid"})
    )
    
    # Last item
    last_items = (
        df
        .sort("ts")
        .group_by("session", maintain_order=True)
        .last()
        .select(["session", "aid"])
        .rename({"aid": "last_aid"})
    )
    
    return session_feats, interaction_feats, last_items


def add_all_features(
    candidates_df: pl.DataFrame,
    feature_source_df: pl.DataFrame,
    global_item_feats_all: pl.DataFrame,
    global_item_feats_7d: pl.DataFrame,
) -> pl.DataFrame:
    """Tạo và ghép nối toàn bộ các đặc trưng cho danh sách ứng viên."""
    df = candidates_df.clone()
    
    # Ghép global item features
    df = df.join(global_item_feats_all, on="candidate_aid", how="left")
    df = df.join(global_item_feats_7d, on="candidate_aid", how="left")
    
    # Tính toán session features cho các session đang xét
    active_sessions = df["session"].unique()
    session_history = feature_source_df.filter(pl.col("session").is_in(active_sessions))
    
    session_feats, interaction_feats, last_items = create_session_features(session_history)
    
    df = df.join(session_feats, on="session", how="left")
    df = df.join(interaction_feats, on=["session", "candidate_aid"], how="left")
    df = df.join(last_items, on="session", how="left")
    
    # Điền giá trị mặc định cho rank (999) và weight (0.0)
    for c in ["rank_clicks", "rank_cart_order", "rank_buy2buy"]:
        if c not in df.columns:
            df = df.with_columns(pl.lit(999).cast(pl.UInt16).alias(c))
        else:
            df = df.with_columns(pl.col(c).fill_null(999))
            
    for c in ["wgt_clicks", "wgt_cart_order", "wgt_buy2buy"]:
        if c not in df.columns:
            df = df.with_columns(pl.lit(0.0).cast(pl.Float32).alias(c))
        else:
            df = df.with_columns(pl.col(c).fill_null(0.0))
            
    # Tính toán đặc trưng phái sinh (Derived features)
    df = df.with_columns([
        # Click/Order trend
        (pl.col("item_click_cnt_7d").fill_null(0) / (pl.col("item_click_cnt_all").fill_null(0) + 10))
            .alias("click_trend_7d"),
        (pl.col("item_order_cnt_7d").fill_null(0) / (pl.col("item_order_cnt_all").fill_null(0) + 10))
            .alias("order_trend_7d"),
        (pl.col("item_buy_ratio_7d").fill_null(0) - pl.col("item_buy_ratio_all").fill_null(0))
            .alias("conversion_trend_diff"),
        
        # Rank diff
        (pl.col("rank_clicks") - pl.col("rank_buy2buy")).cast(pl.Int32).alias("rank_diff_click_b2b"),
        (pl.col("rank_cart_order") - pl.col("rank_buy2buy")).cast(pl.Int32).alias("rank_diff_buy_b2b"),
        
        # Combined weight
        (pl.col("wgt_buy2buy") * 2.0 + pl.col("wgt_cart_order") * 1.0).alias("combined_buy_weight"),
        
        # Recency
        (pl.col("session_end_ts") - pl.col("last_item_ts")).fill_null(7 * 24 * 3600).alias("recency_sec"),
        
        # Binary flags
        (pl.col("candidate_aid") == pl.col("last_aid")).cast(pl.Int8).fill_null(0).alias("is_last_viewed"),
        (pl.col("num_repetitions").fill_null(0) > 1).cast(pl.Int8).alias("is_repeated"),
        pl.col("source_history").fill_null(0),
    ])
    
    df = df.with_columns(
        pl.col("recency_sec").cast(pl.Float64).log1p().alias("log_recency")
    )
    
    df = df.with_columns([
        (pl.col("wgt_buy2buy") / (pl.col("log_recency") + 1)).alias("wgt_b2b_decayed"),
        (pl.col("wgt_clicks") / (pl.col("log_recency") + 1)).alias("wgt_clicks_decayed"),
    ])
    
    a = pl.col("rank_clicks").cast(pl.Int32)
    b = pl.col("rank_cart_order").cast(pl.Int32)
    c = pl.col("rank_buy2buy").cast(pl.Int32)
    
    min_val = pl.min_horizontal([a, b, c])
    max_val = pl.max_horizontal([a, b, c])
    
    df = df.with_columns([
        min_val.cast(pl.UInt16).alias("min_rank_1"),
        (a + b + c - min_val - max_val).cast(pl.UInt16).alias("min_rank_2"),
        ((a < 999).cast(pl.Int8) + (b < 999).cast(pl.Int8) + (c < 999).cast(pl.Int8)).alias("n_sources_present")
    ])
    
    cols_to_drop = ["session_end_ts", "last_item_ts", "first_item_ts", "last_aid"]
    df = df.drop([c for c in cols_to_drop if c in df.columns])
    df = df.fill_null(0)
    
    return df

# ══════════════════════════════════════════════════════════════════════════════
# 3. Hàm Inference Chính (recommend_for_batch)
# ══════════════════════════════════════════════════════════════════════════════

def recommend_for_batch(
    session_events: pl.DataFrame,
    covisit_clicks: pl.DataFrame,
    covisit_cart_order: pl.DataFrame,
    covisit_buy2buy: pl.DataFrame,
    popular_items: pl.DataFrame,
    global_item_feats_all: pl.DataFrame,
    global_item_feats_7d: pl.DataFrame,
    models: dict,
    feature_names: dict,
) -> dict:
    """
    Sinh gợi ý top 20 (clicks, carts, orders) cho từng session trong lô đầu vào.
    
    Args:
        session_events: DataFrame chứa lịch sử (session, aid, ts, type).
        covisit_clicks, covisit_cart_order, covisit_buy2buy: Ma trận đồng truy cập.
        popular_items: Danh sách item phổ biến làm fallback.
        global_item_feats_all, global_item_feats_7d: Các đặc trưng item toàn cục.
        models: Dictionary chứa 3 model LGBMRanker đã train.
        feature_names: Dictionary chứa danh sách tên features của từng model.
        
    Returns:
        dict: { session_id: { "clicks": [aid1, ...], "carts": [...], "orders": [...] } }
    """
    sessions = session_events["session"].unique().to_list()
    results = {sess: {"clicks": [], "carts": [], "orders": []} for sess in sessions}
    
    # 1. Lấy thông tin history độc nhất
    history = session_events.select(["session", "aid"]).unique()
    
    # 2. Sinh ứng viên (Retrieval)
    candidates = generate_candidates(
        history,
        covisit_clicks,
        covisit_cart_order,
        covisit_buy2buy,
        popular_items,
        chunk_size=max(len(sessions), 1)
    )
    
    if candidates.height == 0:
        return results
        
    # 3. Tạo đặc trưng (Feature Engineering)
    featured = add_all_features(
        candidates,
        session_events,
        global_item_feats_all,
        global_item_feats_7d,
    )
    
    if featured.height == 0:
        return results
        
    # 4. Dự đoán điểm số và xếp hạng top 20
    for pred_type in ["clicks", "carts", "orders"]:
        feats = feature_names[pred_type]
        
        # Đảm bảo các đặc trưng đầu vào mô hình yêu cầu đều tồn tại
        for f in feats:
            if f not in featured.columns:
                featured = featured.with_columns(pl.lit(0.0).alias(f))
                
        X = featured.select(feats).to_numpy()
        scores = models[pred_type].predict(X)
        
        preds_df = featured.select(["session", "candidate_aid"]).with_columns(
            pl.Series("score", scores)
        )
        
        top_20 = (
            preds_df
            .sort("score", descending=True)
            .group_by("session", maintain_order=False)
            .head(20)
            .group_by("session")
            .agg(pl.col("candidate_aid").alias("aids"))
        )
        
        for row in top_20.iter_rows():
            sess_id, aids = row
            if sess_id in results:
                results[sess_id][pred_type] = list(aids)
                
    return results


def recommend_topk(
    click_sequence: list,
    covisit_clicks: pl.DataFrame,
    covisit_cart_order: pl.DataFrame,
    covisit_buy2buy: pl.DataFrame,
    popular_items: pl.DataFrame,
    global_item_feats_all: pl.DataFrame,
    global_item_feats_7d: pl.DataFrame,
    models: dict,
    feature_names: dict,
    type_sequence: list = None,
    ts_sequence: list = None,
    target_type: str = "clicks",
    k: int = 20,
    exclude_clicked: bool = True
) -> tuple:
    """
    Sinh gợi ý Top-K sản phẩm cho một session duy nhất từ chuỗi click và các hành động khác.
    
    Args:
        click_sequence: Danh sách các aid gốc tương tác trong session (ví dụ: [123, 456, ...]).
        covisit_clicks, covisit_cart_order, covisit_buy2buy: Ma trận đồng truy cập.
        popular_items: Danh sách item phổ biến làm fallback.
        global_item_feats_all, global_item_feats_7d: Các đặc trưng item toàn cục.
        models: Dictionary chứa 3 model LGBMRanker đã train.
        feature_names: Dictionary chứa danh sách tên features của từng model.
        type_sequence: Danh sách loại hành động tương ứng với click_sequence (0: click, 1: cart, 2: order/buy).
                       Nếu None, mặc định tất cả hành động là 0 (click).
        ts_sequence: Danh sách timestamp tương ứng với click_sequence.
                     Nếu None, mặc định sinh tự động tăng dần.
        target_type: Loại mô hình muốn gợi ý ("clicks", "carts", hoặc "orders").
        k: Số lượng gợi ý muốn lấy.
        exclude_clicked: Nếu True, sẽ lọc bỏ các item trong click_sequence khỏi kết quả gợi ý.
        
    Returns:
        tuple: (top_aids: list, top_scores: list)
    """
    if not click_sequence:
        # Fallback trả về popular items
        pop_aids = popular_items["candidate_aid"].head(k).to_list()
        return pop_aids, [0.0] * len(pop_aids)
        
    n_events = len(click_sequence)
    if type_sequence is None:
        type_sequence = [0] * n_events
    elif len(type_sequence) != n_events:
        raise ValueError("Độ dài type_sequence phải bằng độ dài click_sequence.")
        
    if ts_sequence is None:
        ts_sequence = list(range(n_events))
    elif len(ts_sequence) != n_events:
        raise ValueError("Độ dài ts_sequence phải bằng độ dài click_sequence.")
        
    # 1. Tạo session_events DataFrame giả lập cho 1 session duy nhất
    session_events = pl.DataFrame({
        "session": [0] * n_events,
        "aid": click_sequence,
        "ts": ts_sequence,
        "type": type_sequence
    })
    
    # 2. History unique
    history = session_events.select(["session", "aid"]).unique()
    
    # 3. Sinh ứng viên
    candidates = generate_candidates(
        history,
        covisit_clicks,
        covisit_cart_order,
        covisit_buy2buy,
        popular_items,
        chunk_size=1
    )
    
    # Fallback 1: Không tìm thấy ứng viên
    if candidates.height == 0:
        pop_aids = popular_items["candidate_aid"].to_list()
        if exclude_clicked:
            pop_aids = [aid for aid in pop_aids if aid not in click_sequence]
        top_aids = pop_aids[:k]
        return top_aids, [0.0] * len(top_aids)
        
    # 4. Tính đặc trưng
    featured = add_all_features(
        candidates,
        session_events,
        global_item_feats_all,
        global_item_feats_7d
    )
    
    # Fallback 2: Tính đặc trưng rỗng
    if featured.height == 0:
        pop_aids = popular_items["candidate_aid"].to_list()
        if exclude_clicked:
            pop_aids = [aid for aid in pop_aids if aid not in click_sequence]
        top_aids = pop_aids[:k]
        return top_aids, [0.0] * len(top_aids)
        
    # 5. Điền cột đặc trưng thiếu và Predict
    feats = feature_names[target_type]
    for f in feats:
        if f not in featured.columns:
            featured = featured.with_columns(pl.lit(0.0).alias(f))
            
    X = featured.select(feats).to_numpy()
    scores = models[target_type].predict(X)
    
    # Gom kết quả
    preds_df = featured.select(["candidate_aid"]).with_columns(
        pl.Series("score", scores)
    )
    
    # 6. Loại bỏ các aid đã click trong session
    if exclude_clicked:
        preds_df = preds_df.filter(~pl.col("candidate_aid").is_in(click_sequence))
        
    # 7. Sắp xếp giảm dần và lấy top k
    top_k_df = preds_df.sort("score", descending=True).head(k)
    top_aids = top_k_df["candidate_aid"].to_list()
    top_scores = [float(s) for s in top_k_df["score"].to_list()]
    
    # 8. Điền thêm popular items làm fallback nếu thiếu gợi ý
    if len(top_aids) < k:
        extra_needed = k - len(top_aids)
        pop_aids = popular_items["candidate_aid"].to_list()
        exclude_set = set(top_aids)
        if exclude_clicked:
            exclude_set.update(click_sequence)
        for aid in pop_aids:
            if aid not in exclude_set:
                top_aids.append(aid)
                top_scores.append(-99.0)
                if len(top_aids) == k:
                    break
                    
    return top_aids, top_scores

def recommend_all_targets(
    click_sequence: list,
    covisit_clicks: pl.DataFrame,
    covisit_cart_order: pl.DataFrame,
    covisit_buy2buy: pl.DataFrame,
    popular_items: pl.DataFrame,
    global_item_feats_all: pl.DataFrame,
    global_item_feats_7d: pl.DataFrame,
    models: dict,
    feature_names: dict,
    type_sequence: list = None,
    ts_sequence: list = None,
    k: int = 20,
    exclude_clicked: bool = True
) -> dict:
    """
    Sinh gợi ý đồng thời cho cả 3 target ("clicks", "carts", "orders") 
    của một session duy nhất để tránh việc phải lặp thủ công ở bên ngoài.
    
    Returns:
        dict: {
            "clicks": [...],
            "carts": [...],
            "orders": [...]
        }
    """
    targets = ["clicks", "carts", "orders"]
    result = {}
    
    for target in targets:
        top_aids, _ = recommend_topk(
            click_sequence=click_sequence,
            covisit_clicks=covisit_clicks,
            covisit_cart_order=covisit_cart_order,
            covisit_buy2buy=covisit_buy2buy,
            popular_items=popular_items,
            global_item_feats_all=global_item_feats_all,
            global_item_feats_7d=global_item_feats_7d,
            models=models,
            feature_names=feature_names,
            type_sequence=type_sequence,
            ts_sequence=ts_sequence,
            target_type=target,
            k=k,
            exclude_clicked=exclude_clicked
        )
        result[target] = top_aids
        
    return result

# SASRec Base

In [7]:
def multiply_head_with_embedding(prediction_head, embeddings):
    return prediction_head.matmul(embeddings.transpose(-1, -2))

def bce_loss(pos_logits, neg_logits, mask, epsilon=1e-10, device="cpu"):
    loss = torch.log(1. + torch.exp(-pos_logits) + epsilon) + torch.log(1. + torch.exp(neg_logits) + epsilon).mean(-1, keepdim=True)
    return (loss * mask.unsqueeze(-1)).sum() / mask.sum()

class DynamicPositionEmbedding(torch.nn.Module):
    def __init__(self, max_len, dimension):
        super(DynamicPositionEmbedding, self).__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(max_len, dimension)
        self.pos_indices = torch.arange(0, self.max_len, dtype=torch.int)
        self.register_buffer('pos_indices_const', self.pos_indices)

    def forward(self, x, device='cpu'):
        seq_len = x.shape[1]
        return self.embedding(self.pos_indices_const[-seq_len:]) + x

class SASRec(py_light.LightningModule):
    def __init__(self, hidden_size, dropout_rate, max_len, num_items, batch_size,
                 sampling_style, topk_sampling=False, topk_sampling_k=1000,
                 learning_rate=0.001, num_layers=2, loss='bce', bpr_penalty=None,
                 optimizer='adam', output_bias=False, share_embeddings=True,
                 final_activation=False):
        super(SASRec, self).__init__()
        self.learning_rate = learning_rate
        self.hidden_size = hidden_size
        self.dropout_rate = dropout_rate
        self.num_items = num_items
        self.batch_size = batch_size
        self.num_layers = num_layers
        self.max_len = max_len
        self.output_bias = output_bias
        self.share_embeddings = share_embeddings
        self.future_mask = torch.triu(torch.ones(max_len, max_len) * float('-inf'), diagonal=1)
        self.register_buffer('future_mask_const', self.future_mask)
        self.register_buffer('seq_diag_const', ~diag(torch.ones(max_len, dtype=torch.bool)))
        self.register_buffer('bias_ones', torch.ones([self.batch_size, self.max_len, 1]))
        if output_bias and share_embeddings:
            self.item_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.item_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)
        self.positional_embedding_layer = DynamicPositionEmbedding(max_len, hidden_size)
        torch.nn.init.xavier_uniform_(self.item_embedding.weight.data)
        torch.nn.init.xavier_uniform_(self.positional_embedding_layer.embedding.weight.data)
        if share_embeddings:
            self.output_embedding = self.item_embedding
        elif (not share_embeddings) and output_bias:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)
        self.norm = nn.LayerNorm([hidden_size])
        self.input_dropout = Dropout(dropout_rate)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=1,
                                                    dim_feedforward=hidden_size,
                                                    dropout=dropout_rate,
                                                    batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers, norm=self.norm)
        self.merge_attn_mask = True
        self.final_activation = nn.ELU(0.5) if final_activation else nn.Identity()
        self.loss_fn = loss
        if self.loss_fn == 'bce':
            self.loss = bce_loss
        else:
            raise ValueError('Loss function not supported')
        self.sampling_style = sampling_style
        self.topk_sampling = topk_sampling
        self.topk_sampling_k = topk_sampling_k
        self.optimizer = optimizer
        self.save_hyperparameters()

    def merge_attn_masks(self, padding_mask):
        batch_size = padding_mask.shape[0]
        seq_len = padding_mask.shape[1]
        if not self.merge_attn_mask:
            return self.future_mask_const[:seq_len, :seq_len]
        padding_mask_broadcast = ~padding_mask.bool().unsqueeze(1)
        future_masks = tile(self.future_mask_const[:seq_len, :seq_len], (batch_size, 1, 1))
        merged_masks = logical_or(padding_mask_broadcast, future_masks)
        diag_masks = tile(self.seq_diag_const[:seq_len, :seq_len], (batch_size, 1, 1))
        return logical_and(diag_masks, merged_masks)

    def forward(self, item_indices, mask):
        att_mask = self.merge_attn_masks(mask)
        items = self.item_embedding(item_indices)[:, :, :-1] if self.output_bias and self.share_embeddings \
                else self.item_embedding(item_indices)
        x = items * np.sqrt(self.hidden_size)
        x = self.positional_embedding_layer(x)
        x = self.encoder(self.input_dropout(x), att_mask)
        return concat([x, self.bias_ones], dim=-1) if self.output_bias else x

    def configure_optimizers(self):
        if self.optimizer == 'adam':
            return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        elif self.optimizer == 'adagrad':
            return torch.optim.Adagrad(self.parameters(), lr=self.learning_rate)
        raise ValueError('Optimizer not supported')

class SASRecInference:
    def __init__(self, checkpoint_path, device=None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.model = SASRec.load_from_checkpoint(checkpoint_path, map_location=device)
        self.model.eval()
        self.model.to(device)
        self.max_len = self.model.max_len
        self.num_items = self.model.num_items
        print(f"[SASRec] Loaded | device={device} | max_len={self.max_len} | num_items={self.num_items}")

    def _prepare_sequence(self, click_sequence: list) -> tuple:
        seq = click_sequence[-self.max_len:]
        seq_len = len(seq)
        pad_len = self.max_len - seq_len
        padded = [0] * pad_len + seq
        mask_arr = [0.0] * pad_len + [1.0] * seq_len
        item_indices = torch.tensor([padded], dtype=torch.long, device=self.device)
        mask = torch.tensor([mask_arr], dtype=torch.float, device=self.device)
        return item_indices, mask

    @torch.no_grad()
    def recommend_topk(self, click_sequence: list, k: int = 60, exclude_clicked: bool = True) -> tuple:
        item_indices, mask = self._prepare_sequence(click_sequence)
        x_hat = self.model.forward(item_indices, mask)
        last_hidden = x_hat[:, -1, :]
        logits = multiply_head_with_embedding(last_hidden, self.model.output_embedding.weight)
        logits = logits.squeeze(0)
        logits[0] = float("-inf")
        if exclude_clicked:
            for aid in click_sequence:
                if 1 <= aid <= self.num_items:
                    logits[aid] = float("-inf")
        actual_k = min(k, self.num_items)
        top_scores_tensor, top_indices_tensor = torch.topk(logits, k=actual_k)
        top_aids_shifted = top_indices_tensor.cpu().tolist()
        top_scores = top_scores_tensor.cpu().tolist()
        top_aids = [aid - 1 for aid in top_aids_shifted]  # về aid gốc
        return top_aids, top_scores

    def split_topk(self, click_sequence: list, k: int = 20, exclude_clicked: bool = True):
        top_aids, top_scores = self.recommend_topk(click_sequence=click_sequence, k=k*3, exclude_clicked=exclude_clicked,)
        # split
        clicks_aids = top_aids[:k]
        carts_aids = top_aids[k: 2*k]
        orders_aids = top_aids[2*k: 3*k]

        return {
            "clicks": clicks_aids,
            "carts": carts_aids,
            "orders": orders_aids,
        }

# Load model

In [8]:
SASREC_BASE_PATH = "/kaggle/input/models/sandaria/sasrec/pytorch/otto-dataset/1/sasrec-otto-epoch4-recall_cutoff_200.308.ckpt"
ARTIFACTS_PATH = "/kaggle/input/notebooks/hngphongkiu/covisited-lgbmranker/artifacts"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HOST = "0.0.0.0"
PORT = 8000

#load sasrec base
sasrec_base = SASRecInference(SASREC_BASE_PATH, device=DEVICE)

#load lbm
(
    covisit_clicks,
    covisit_cart_order,
    covisit_buy2buy,
    popular_items,
    global_item_feats_all,
    global_item_feats_7d,
    models,
    feature_names
) = load_pipeline_artifacts(ARTIFACTS_PATH)

[SASRec] Loaded | device=cuda | max_len=50 | num_items=1855603

[Inference] Đang tải các tài nguyên từ /kaggle/input/notebooks/hngphongkiu/covisited-lgbmranker/artifacts...
[Inference] Tải tài nguyên thành công trong 2.2s!


# Chạy thử kết quả

In [9]:
# test thử cái xem model chạy đc trên kaggle chưa
sample_aid_goc = [1492293,910862,1491172,424964,1515526,440486,109488,1507622,1734061,854637,718983,215311,718983,711125,50049,105393,959544,1734061,1842593,1464360,207905,1628317]
sample_aid_updated = [id + 1 for id in sample_aid_goc]

result = sasrec_base.split_topk(
        click_sequence=sample_aid_updated,
        k=20,
        exclude_clicked=True,
    )
print(f"Input clicks: {sample_aid_goc}")
print(result)

Input clicks: [1492293, 910862, 1491172, 424964, 1515526, 440486, 109488, 1507622, 1734061, 854637, 718983, 215311, 718983, 711125, 50049, 105393, 959544, 1734061, 1842593, 1464360, 207905, 1628317]
{'clicks': [556314, 1123537, 464673, 1007803, 1763549, 103974, 880854, 1491502, 1453277, 937893, 165854, 259724, 1245311, 606921, 1781681, 1532558, 434686, 1568162, 924095, 1483294], 'carts': [1744773, 662503, 1225099, 495284, 458192, 119351, 317396, 771517, 415990, 1712906, 615357, 939040, 50154, 1778368, 600357, 1596235, 783067, 803406, 283753, 1136088], 'orders': [443337, 1276698, 1835202, 627142, 60342, 1655189, 1285712, 145078, 921561, 235670, 1519786, 1167593, 1228666, 37272, 1796520, 24431, 1787052, 924224, 1533364, 1475928]}


In [15]:
# Chuỗi click mẫu để chạy thử
click_sequence = [1492293, 910862, 1491172, 424964]
type_sequence  = [0, 0, 1, 2] # 2 click, 1 cart, 1 buy
ts_sequence = [1, 2,3, 4]

result = recommend_all_targets(
    click_sequence=click_sequence,
    covisit_clicks=covisit_clicks,
    covisit_cart_order=covisit_cart_order,
    covisit_buy2buy=covisit_buy2buy,
    popular_items=popular_items,
    global_item_feats_all=global_item_feats_all,
    global_item_feats_7d=global_item_feats_7d,
    models=models,
    feature_names=feature_names,
    ts_sequence = ts_sequence,
    type_sequence=type_sequence,
    k=20,
    exclude_clicked=True
)
print(f"\nChạy thử recommend_topk với click_sequence: {click_sequence} và type_sequence: {type_sequence}")
print(result)


Chạy thử recommend_topk với click_sequence: [1492293, 910862, 1491172, 424964] và type_sequence: [0, 0, 1, 2]
{'clicks': [1535600, 1356026, 574185, 1816941, 1270398, 273740, 1768728, 765983, 1661379, 296738, 1035080, 728014, 406364, 1778526, 607735, 855107, 554352, 432989, 899354, 1125638], 'carts': [1535600, 1768728, 1816941, 1270398, 406364, 1356026, 728014, 273740, 1221201, 1661379, 296738, 232609, 571331, 149662, 192687, 1778526, 931515, 855107, 228090, 1604385], 'orders': [1535600, 1270398, 1768728, 1221201, 1816941, 296738, 1604385, 125278, 1356026, 855107, 273740, 1445562, 1634470, 1022566, 1033617, 728014, 232609, 1778526, 166037, 966841]}


# FastAPI Server

In [11]:
# request/response schemas 
class RecommendRequest(BaseModel):
    click_sequence: List[int] = Field(..., min_length=1, description="List aid gốc (chưa +1)")
    type_sequence: Optional[List[str]] = Field(None)
    ts_sequence: Optional[List[int]] = Field(None)
    k: int = Field(20, ge=1, le=500, description="Số lượng gợi ý trả về")
    exclude_clicked: bool = Field(True, description="Loại bỏ item đã click khỏi kết quả")
    model: Literal["sasrec_base", "lbmrerank"] = Field("lbmrerank")
    
class RecommendResponse(BaseModel):
    recommend:  dict  
    model_used: str

In [12]:
app = FastAPI(title="SASRec Recommendation API")

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE}

@app.post("/recommend", response_model=RecommendResponse)
def recommend(req: RecommendRequest):
    try:
        if req.model == "sasrec_base":
            shifted = [aid + 1 for aid in req.click_sequence]
            result = sasrec_base.split_topk(
                click_sequence=shifted,
                k=req.k,
                exclude_clicked=req.exclude_clicked,
            )
        elif req.model == "lbmrerank":
            type_seq_int = map_type_sequence(req.type_sequence)
            result = recommend_all_targets(
                click_sequence=req.click_sequence,
                covisit_clicks=covisit_clicks,
                covisit_cart_order=covisit_cart_order,
                covisit_buy2buy=covisit_buy2buy,
                popular_items=popular_items,
                global_item_feats_all=global_item_feats_all,
                global_item_feats_7d=global_item_feats_7d,
                models=models,
                feature_names=feature_names,
                type_sequence=type_seq_int,
                ts_sequence=req.ts_sequence,
                k=req.k,
                exclude_clicked=req.exclude_clicked,
            )
        else:
            raise HTTPException(status_code=400, detail=f"Model không hợp lệ: {req.model}")

        return RecommendResponse(
            recommend=result,
            model_used=req.model,
        )
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Start server and Cloudflare tunnel

In [ ]:
import os
import re
import subprocess

def start_cloudflare_tunnel(port):
    print("[CLOUDFLARE] Đang khởi tạo tunnel...")
    if not os.path.exists("./cloudflared"):
        print("[CLOUDFLARE] Đang tải cloudflared...")
        subprocess.run(["wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", "cloudflared"])
        subprocess.run(["chmod", "+x", "cloudflared"])

    proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in iter(proc.stdout.readline, ""):
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print(f"\n{'='*60}")
            print(f"  SASRec API URL: {url}")
            print(f"  Docs:           {url}/docs")
            print(f"{'='*60}\n")
            return url, proc
    return None, proc

# Chạy Uvicorn trong thread riêng
server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host=HOST, port=PORT, log_level="warning"),
    daemon=True
)
server_thread.start()
time.sleep(3)

public_url, tunnel_proc = start_cloudflare_tunnel(PORT)

if public_url:
    while True:
        time.sleep(10)
else:
    print("[ERROR] Không thể khởi tạo Cloudflare Tunnel.")

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


[CLOUDFLARE] Đang khởi tạo tunnel...

  SASRec API URL: https://opera-theme-carry-lawn.trycloudflare.com
  Docs:           https://opera-theme-carry-lawn.trycloudflare.com/docs

